In [1]:
import os
import sys

import numpy as np
from scipy import sparse
import pandas as pd


In [3]:
dataset = "ml-20m/ratings.csv"
output_dir = "2step"
threshold = 3.5
min_uc = 5
min_sc = 0
n_heldout_users = 10000

First, download the dataset at http://files.grouplens.org/datasets/movielens/ml-20m.zip

In [4]:
raw_data = pd.read_csv(dataset, header=0)

In [5]:
raw_data = raw_data[raw_data['rating'] > threshold]

In [6]:
raw_data.head()

,userId,movieId,rating,timestamp
6,1,151,4.0,1094785734
7,1,223,4.0,1112485573
8,1,253,4.0,1112484940
9,1,260,4.0,1112484826
10,1,293,4.0,1112484703


### Data splitting procedure

In [7]:
def get_count(tp, id):
    playcount_groupbyid = tp[[id]].groupby(id, as_index=False)
    count = playcount_groupbyid.size()
    return count

In [8]:
def filter_triplets(tp, min_uc=min_uc, min_sc=min_sc):
    # Only keep the triplets for items which were clicked on by at least min_sc users. 
    if min_sc > 0:
        itemcount = get_count(tp, 'movieId')
        tp = tp[tp['movieId'].isin(itemcount.index[itemcount >= min_sc])]
    
    # Only keep the triplets for users who clicked on at least min_uc items
    # After doing this, some of the items will have less than min_uc users, but should only be a small proportion
    if min_uc > 0:
        usercount = get_count(tp, 'userId')
        tp = tp[tp['userId'].isin(usercount.index[usercount >= min_uc])]
    
    # Update both usercount and itemcount after filtering
    usercount, itemcount = get_count(tp, 'userId'), get_count(tp, 'movieId') 
    return tp, usercount, itemcount

Only keep items that are clicked on by at least 5 users

In [9]:
raw_data, user_activity, item_popularity = filter_triplets(raw_data)

In [10]:
sparsity = 1. * raw_data.shape[0] / (user_activity.shape[0] * item_popularity.shape[0])

print("After filtering, there are %d watching events from %d users and %d movies (sparsity: %.3f%%)" % 
      (raw_data.shape[0], user_activity.shape[0], item_popularity.shape[0], sparsity * 100))

After filtering, there are 9990682 watching events from 136677 users and 20720 movies (sparsity: 0.353%)


In [11]:
unique_uid = user_activity.index

np.random.seed(98765)
idx_perm = np.random.permutation(unique_uid.size)
unique_uid = unique_uid[idx_perm]

In [12]:
# create train/validation/test users
n_users = unique_uid.size
n_train_users = (n_users - n_heldout_users * 2) // 2

tr1_users = unique_uid[:n_train_users]
tr2_users = unique_uid[n_train_users:(n_users - n_heldout_users * 2)]
vd_users = unique_uid[(n_users - n_heldout_users * 2): (n_users - n_heldout_users)]
te_users = unique_uid[(n_users - n_heldout_users):]

In [13]:
train1_plays = raw_data.loc[raw_data['userId'].isin(tr1_users)]

In [14]:
unique_sid = pd.unique(train1_plays['movieId'])

In [15]:
show2id = dict((sid, i) for (i, sid) in enumerate(unique_sid))
profile2id = dict((pid, i) for (i, pid) in enumerate(unique_uid))

In [16]:
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

with open(os.path.join(output_dir, 'unique_sid.txt'), 'w') as f:
    for sid in unique_sid:
        f.write('%s\n' % sid)
        
with open(os.path.join(output_dir, 'unique_uid.txt'), 'w') as f:
    for uid in unique_uid:
        f.write('%s\n' % uid)

In [17]:
def split_train_test_proportion(data, test_prop=0.2):
    data_grouped_by_user = data.groupby('userId')
    tr_list, te_list = list(), list()

    np.random.seed(98765)

    for i, (_, group) in enumerate(data_grouped_by_user):
        group = group.sort_values('timestamp')
        n_items_u = len(group)

        if n_items_u >= 5:
#             idx = np.zeros(n_items_u, dtype='bool')
#             idx[np.random.choice(n_items_u, size=int(test_prop * n_items_u), replace=False).astype('int64')] = True
            size=int(test_prop * n_items_u)

            tr_list.append(group[:-size])
            te_list.append(group[-size:])
        else:
            tr_list.append(group)

        if i % 1000 == 0:
            print("%d users sampled" % i)
            sys.stdout.flush()

    data_tr = pd.concat(tr_list)
    data_te = pd.concat(te_list)
    
    return data_tr, data_te

# split_train_test_proportion(vad_plays)

In [18]:
train2_plays = raw_data.loc[raw_data['userId'].isin(tr2_users)]
train2_plays = train2_plays.loc[train2_plays['movieId'].isin(unique_sid)]

In [19]:
train2_plays

,userId,movieId,rating,timestamp
6,1,151,4.0,1094785734
7,1,223,4.0,1112485573
8,1,253,4.0,1112484940
9,1,260,4.0,1112484826
10,1,293,4.0,1112484703
...,...,...,...,...
19999630,138489,2959,4.5,1352990114
19999631,138489,3671,4.0,1352989107
19999632,138489,4973,4.0,1352990124
19999633,138489,5291,4.5,1352990158


In [20]:
# train2_plays = lol(train2_plays)

In [21]:
vad_plays = raw_data.loc[raw_data['userId'].isin(vd_users)]
vad_plays = vad_plays.loc[vad_plays['movieId'].isin(unique_sid)]

In [22]:
vad_plays_tr, vad_plays_te = split_train_test_proportion(vad_plays)

0 users sampled
1000 users sampled
2000 users sampled
3000 users sampled
4000 users sampled
5000 users sampled
6000 users sampled
7000 users sampled
8000 users sampled
9000 users sampled


In [23]:
test_plays = raw_data.loc[raw_data['userId'].isin(te_users)]
test_plays = test_plays.loc[test_plays['movieId'].isin(unique_sid)]

In [24]:
test_plays_tr, test_plays_te = split_train_test_proportion(test_plays)

0 users sampled
1000 users sampled
2000 users sampled
3000 users sampled
4000 users sampled
5000 users sampled
6000 users sampled
7000 users sampled
8000 users sampled
9000 users sampled


### Save the data into (user_index, item_index) format

In [25]:
def numerize(tp):
    uid = list(map(lambda x: profile2id[x], tp['userId']))
    sid = list(map(lambda x: show2id[x], tp['movieId']))
    return pd.DataFrame(data={'uid': uid, 'sid': sid}, columns=['uid', 'sid'])

In [26]:
train1_data = numerize(train1_plays)
train1_data.to_csv(os.path.join(output_dir, 'train1.csv'), index=False)

In [27]:
train2_data = numerize(train2_plays)
train2_data.to_csv(os.path.join(output_dir, 'train2.csv'), index=False)

In [28]:
vad_data_tr = numerize(vad_plays_tr)
vad_data_tr.to_csv(os.path.join(output_dir, 'validation_tr.csv'), index=False)

In [29]:
vad_data_te = numerize(vad_plays_te)
vad_data_te.to_csv(os.path.join(output_dir, 'validation_te.csv'), index=False)

In [30]:
test_data_tr = numerize(test_plays_tr)
test_data_tr.to_csv(os.path.join(output_dir, 'test_tr.csv'), index=False)

In [31]:
test_data_te = numerize(test_plays_te)
test_data_te.to_csv(os.path.join(output_dir, 'test_te.csv'), index=False)

In [38]:
# def load_train_data(csv_file, n_items, n_users, global_indexing=False):
#     tp = pd.read_csv(csv_file)
    
#     n_users = n_users if global_indexing else tp['uid'].max() + 1

#     rows, cols = tp['uid'], tp['sid']
#     data = sparse.csr_matrix((np.ones_like(rows),
#                              (rows, cols)), dtype='float64',
#                              shape=(n_users, n_items))
#     return data

def load_train_data(csv_file, n_items, n_users, global_indexing=False):
    tp = pd.read_csv(csv_file)

    if global_indexing:
        start_idx = 0
        end_idx = len(unique_uid) - 1
    else:
        start_idx = tp['uid'].min()
        end_idx = tp['uid'].max()

    rows, cols = tp['uid'] - start_idx, tp['sid']

    data = sparse.csr_matrix((np.ones_like(rows),
                             (rows, cols)), dtype='float64', shape=(end_idx - start_idx + 1, n_items))
    return data

def load_tr_te_data(csv_file_tr, csv_file_te, n_items, n_users, global_indexing=False):
    tp_tr = pd.read_csv(csv_file_tr)
    tp_te = pd.read_csv(csv_file_te)

    if global_indexing:
        start_idx = 0
        end_idx = len(unique_uid) - 1
    else:
        start_idx = min(tp_tr['uid'].min(), tp_te['uid'].min())
        end_idx = max(tp_tr['uid'].max(), tp_te['uid'].max())

    rows_tr, cols_tr = tp_tr['uid'] - start_idx, tp_tr['sid']
    rows_te, cols_te = tp_te['uid'] - start_idx, tp_te['sid']

    data_tr = sparse.csr_matrix((np.ones_like(rows_tr),
                             (rows_tr, cols_tr)), dtype='float64', shape=(end_idx - start_idx + 1, n_items))
    data_te = sparse.csr_matrix((np.ones_like(rows_te),
                             (rows_te, cols_te)), dtype='float64', shape=(end_idx - start_idx + 1, n_items))
    return data_tr, data_te


def get_data(dataset, global_indexing=False):
    unique_sid = list()
    with open(os.path.join(dataset, 'unique_sid.txt'), 'r') as f:
        for line in f:
            unique_sid.append(line.strip())
    
    unique_uid = list()
    with open(os.path.join(dataset, 'unique_uid.txt'), 'r') as f:
        for line in f:
            unique_uid.append(line.strip())
            
    n_items = len(unique_sid)
    n_users = len(unique_uid)
    
    train1_data = load_train_data(os.path.join(dataset, 'train1.csv'), n_items, n_users, global_indexing=global_indexing)
    train2_data = load_train_data(os.path.join(dataset, 'train2.csv'), n_items, n_users, global_indexing=global_indexing)


    vad_data_tr, vad_data_te = load_tr_te_data(os.path.join(dataset, 'validation_tr.csv'),
                                               os.path.join(dataset, 'validation_te.csv'),
                                               n_items, n_users, 
                                               global_indexing=global_indexing)

    test_data_tr, test_data_te = load_tr_te_data(os.path.join(dataset, 'test_tr.csv'),
                                                 os.path.join(dataset, 'test_te.csv'),
                                                 n_items, n_users, 
                                                 global_indexing=global_indexing)
    
    data = train1_data, train2_data, vad_data_tr, vad_data_te, test_data_tr, test_data_te
    data = (x.astype('float32') for x in data)
    
    return data

In [39]:
data = get_data(output_dir)
train1_data, train2_data, valid_in_data, valid_out_data, test_in_data, test_out_data = data


In [40]:
from scipy.sparse import save_npz

save_npz(output_dir + '/train1_data.npz', train1_data)
save_npz(output_dir + '/train2_data.npz', train2_data)
save_npz(output_dir + '/valid_in_data.npz', valid_in_data)
save_npz(output_dir + '/valid_out_data.npz', valid_out_data)
save_npz(output_dir + '/test_in_data.npz', test_in_data)
save_npz(output_dir + '/test_out_data.npz', test_out_data)

In [41]:
train1_data.sum(), train2_data.sum(), train1_data.shape, train2_data.shape

(4272898.0, 4262533.0, (58338, 17691), (58339, 17691))

In [42]:
train1_data.sum(), train2_data.sum(), train1_data.shape, train2_data.shape

(4272898.0, 4262533.0, (58338, 17691), (58339, 17691))